# Deep Hedging — Phase 3 : le couvreur neuronal

On remplace la règle du delta par un réseau `F_theta(état du marché)` qui sort la position à tenir. On déroule la couverture sur toute la trajectoire, et on minimise la **CVaR** de la perte finale par rapport aux poids `theta`, sous coûts de transaction.

Données infinies : on simule des trajectoires **fraîches à chaque itération**, donc pas de surapprentissage au sens classique. La perte est la forme RU de la brique 5.

**Cibles** (validées en numpy) : le delta-hedging pur donne CVaR $\approx 6.40$, la meilleure bande fixe $\approx 5.70$. Le réseau doit battre 6.40 et viser $\le 5.70$.

In [ ]:
import torch
import numpy as np
from scipy.stats import norm
torch.manual_seed(0)
print("torch", torch.__version__)

## Simulation, prime, et perte CVaR (en torch)

In [ ]:
S0, K, mu, r, sigma, T = 100., 100., 0.10, 0.02, 0.20, 1.0
n, cost, alpha = 63, 0.01, 0.95
dt = T / n
times = torch.linspace(0, T, n + 1)

def simulate_paths(m):
    """m trajectoires GBM, tenseur (m, n+1). Nouvelles à chaque appel."""
    Z = torch.randn(m, n)
    inc = (mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z
    logp = torch.cat([torch.zeros(m, 1), torch.cumsum(inc, dim=1)], dim=1)
    return S0 * torch.exp(logp)

# prime encaissée = prix Black-Scholes du call (constante), calculée en numpy
d1 = (np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T)); d2 = d1 - sigma*np.sqrt(T)
premium = float(S0*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2))

def cvar_ru(loss, w, alpha=0.95):
    return w + torch.mean(torch.relu(loss - w)) / (1.0 - alpha)

## Le réseau

Un MLP partagé entre toutes les dates. Entrées à la date `k` : la log-moneyness `ln(S_k/K)`, le temps restant `tau_k`, et la **position courante** `delta_{k-1}` (c'est elle qui permet au réseau d'apprendre une bande de non-transaction). Sortie : la position `delta_k` à tenir.

In [ ]:
class HedgeNet(torch.nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(3, hidden), torch.nn.ReLU(),
            torch.nn.Linear(hidden, hidden), torch.nn.ReLU(),
            torch.nn.Linear(hidden, 1),
        )
    def forward(self, x):        # x : (m, 3)  ->  (m,)
        return self.net(x).squeeze(-1)

## Dérouler la couverture et calculer la perte

Même comptabilité auto-financée que la brique 3, mais `delta_k` vient du réseau. Tout est différentiable, donc `backward()` propage le gradient de la CVaR jusqu'aux poids.

In [ ]:
def hedging_loss(net, w, m):
    S = simulate_paths(m)
    cash = torch.full((m,), premium)             # prime encaissée
    delta_prev = torch.zeros(m)                   # aucune position au départ
    for k in range(n):
        tau = float(T - times[k].item())
        feat = torch.stack([torch.log(S[:, k]/K),
                            torch.full((m,), tau),
                            delta_prev], dim=1)    # (m, 3)
        delta_k = net(feat)                        # position choisie par le réseau
        trade = delta_k - delta_prev
        cash = cash - trade*S[:, k] - cost*torch.abs(trade)*S[:, k]
        cash = cash * np.exp(r*dt)                 # capitalisation
        delta_prev = delta_k
    payoff = torch.clamp(S[:, -1] - K, min=0.0)
    pnl = cash + delta_prev*S[:, -1] - payoff
    return cvar_ru(-pnl, w, alpha)                 # CVaR de la perte

## Entraînement

Sur CPU, quelques minutes. On peut réduire `n_iter` ou la taille de batch si c'est lent.

In [ ]:
net = HedgeNet()
w = torch.zeros(1, requires_grad=True)
opt = torch.optim.Adam(list(net.parameters()) + [w], lr=1e-3)

n_iter, batch = 2000, 2048
for it in range(n_iter):
    opt.zero_grad()
    loss = hedging_loss(net, w, batch)
    loss.backward()
    opt.step()
    if it % 200 == 0:
        print(f"iter {it:4d}   CVaR (train) = {loss.item():.3f}")

## Évaluation sur trajectoires fraîches

On compare la CVaR du réseau aux cibles : delta pur 6.40, meilleure bande fixe 5.70.

In [ ]:
with torch.no_grad():
    # grande évaluation hors échantillon
    m = 100_000
    S = simulate_paths(m)
    cash = torch.full((m,), premium); delta_prev = torch.zeros(m)
    for k in range(n):
        tau = float(T - times[k].item())
        feat = torch.stack([torch.log(S[:, k]/K), torch.full((m,), tau), delta_prev], dim=1)
        delta_k = net(feat); trade = delta_k - delta_prev
        cash = cash - trade*S[:, k] - cost*torch.abs(trade)*S[:, k]
        cash = cash * np.exp(r*dt); delta_prev = delta_k
    pnl = cash + delta_prev*S[:, -1] - torch.clamp(S[:, -1]-K, min=0.0)
    loss = -pnl
    var = torch.quantile(loss, alpha)
    cvar = loss[loss >= var].mean()
    print(f"CVaR réseau (hors échantillon) = {cvar.item():.3f}")
    print(f"cibles : delta pur 6.40 | meilleure bande fixe 5.70")

## Ce qu'on regarde ensuite

Si le réseau bat 6.40 et s'approche de (ou passe sous) 5.70, la thèse du projet est démontrée : un couvreur appris bat le delta sous frictions. On analysera alors *ce qu'il a appris* (sa position en fonction de S, comparée au delta et à la bande), on testera d'autres coûts et mesures de risque, et on passera à Heston (phase 4).